In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import json
from pathlib import Path

id_attrs = [
    "variable_id",
    "domain_id",
    "driving_source_id",
    "driving_experiment_id",
    "driving_variant_label",
    "institution_id",
    "source_id",
    "version_realization",
    "frequency",
    "version",
]


def filename_to_id(filename):
    """
    Extract the dataset id from the filename.
    """
    stem = Path(filename).stem
    path = str(Path(filename).parent)
    version = path.split("/")[-1]
    values = stem.split("_")[0 : len(id_attrs) - 1] + [version]
    return ".".join(values)


def filename_to_attrs(filename):
    """
    Create a dictionary with the dataset id as key and the filename as value.
    """
    stem = Path(filename).stem
    path = str(Path(filename).parent)
    version = path.split("/")[-1]
    values = stem.split("_")[0 : len(id_attrs) - 1] + [version]
    return dict(zip(id_attrs, values))


with open("report/compliance-report.json") as fp:
    cc_data = json.load(fp)

In [3]:
import pandas as pd

prios = {
    "cf": ["low_priorities", "medium_priorities", "high_priorities"],
    "cc6": ["low_priorities", "medium_priorities", "high_priorities"],
}

cols = ["scored_points", "possible_points", "high_count", "medium_count", "low_count"]

id_attrs = [
    "variable_id",
    "domain_id",
    "driving_source_id",
    "driving_experiment_id",
    "driving_variant_label",
    "institution_id",
    "source_id",
    "version_realization",
    "frequency",
    "version",
]


def concat_messages(tests):
    summary = ""
    for test in tests:
        if test.get("msgs"):
            summary += "\n".join(test["msgs"]) + "\n"
    return summary


def summarize(test, results):
    summaries = {}
    test_id = test.split(":")[0]
    summaries = {f"{test_id}:{c}": results[c] for c in cols}
    for prio in prios[test_id]:
        tests = results.get(prio, [])
        summary = concat_messages(tests)
        summaries[f"{test_id}:{prio}"] = summary
    return summaries


def to_dataframe(cc_data):
    df = (
        pd.DataFrame.from_dict(cc_data, orient="index")
        .reset_index()
        .rename(columns={"index": "filename"})
    )
    return df


result = {}

for filename, tests in cc_data.items():
    summary = {}
    for test, results in tests.items():
        summary.update(summarize(test, results))
    result[filename] = filename_to_attrs(filename) | summary

In [6]:
# corrupt = pd.read_csv("report/corrupt_files.csv")
df = (
    pd.DataFrame.from_dict(result, orient="index")
    .reset_index()
    .rename(columns={"index": "filename"})
)
df.to_csv("report/compliance-report.csv", index=False)
# df = df.merge(corrupt, on="filename", how="left")

In [16]:
for group in df.groupby("institution_id").groups:
    print(group)

CNRM-MF
GERICS
ICTP
KNMI


In [27]:
import os


def human_readable(df):
    """
    Creates a human-readable summary of the dataset.

    Parameters:
    df (pandas.DataFrame): The input DataFrame containing the dataset.

    Returns:
    pandas.DataFrame: A DataFrame with grouped and summarized data.
    """

    index = [
        #  "institution_id",
        "domain_id",
        "source_id",
        "driving_experiment_id",
        "driving_source_id",
        "driving_variant_label",
        "version_realization",
        "variable_id",
        "frequency",
        "version",
        "filename",
    ]
    # cols = [c for c in df.columns if c not in index]
    return df.sort_values(index).set_index(index).fillna("")


def create_excel(filename):
    """
    Creates a human-readable Excel file from the dataset.

    Parameters:
    filename (str): The path to the CSV file containing the dataset.

    Returns:
    str: The path to the created Excel file.
    """
    df = pd.read_csv(filename)

    sheets = {
        institution_id: human_readable(df)
        for institution_id, df in df.groupby("institution_id")
    }

    stem, suffix = os.path.splitext(filename)
    xlsxfile = f"{stem}.xlsx"

    with pd.ExcelWriter(xlsxfile, engine="xlsxwriter") as writer:
        workbook = writer.book
        wrap_format = workbook.add_format(
            {"text_wrap": True, "align": "left", "valign": "top"}
        )
        grey_format = workbook.add_format(
            {"bg_color": "#F2F2F2", "text_wrap": True, "align": "left", "valign": "top"}
        )

        header_format = workbook.add_format(
            {
                "bold": True,
                "text_wrap": True,
                "valign": "top",
                "fg_color": "#D7E4BC",
                "border": 1,
            }
        )
        for sheet_name, sheet_df in sheets.items():
            print(sheet_name)
            sheet_df.to_excel(writer, sheet_name=sheet_name, index=True)
            worksheet = writer.sheets[sheet_name]  # pull worksheet object

            n_index = len(sheet_df.index.names)
            n_rows = len(sheet_df)
            n_cols = len(sheet_df.columns)

            # Write the column headers with the defined format.
            for col_num, value in enumerate(sheet_df.index.names):
                worksheet.write(0, col_num, value, header_format)

            # Write the data columns headers with the defined format.
            for col_num, value in enumerate(sheet_df.columns):
                worksheet.write(0, col_num + n_index, value, header_format)

            # Set wrap for all index columns
            for idx in range(n_index):
                worksheet.set_column(idx, idx, 30, wrap_format)

            # Set wrap for data columns
            for col_num in range(n_index, n_index + n_cols):
                worksheet.set_column(col_num, col_num, 50, wrap_format)

            # Apply alternating row color (starting after header row)
            for row in range(1, n_rows + 1):
                fmt = grey_format if row % 2 == 0 else wrap_format
                # worksheet.set_row(row, 60, fmt)
                # Write data columns
                for col in range(n_cols):
                    value = sheet_df.iloc[row - 1, col]
                    worksheet.write(row, col + n_index, value, fmt)

            # --- Merge repeated MultiIndex cells ---
            # Get the index values as a DataFrame
            idx_df = pd.DataFrame(sheet_df.index.tolist(), columns=sheet_df.index.names)
            start_row = 1  # Excel row index (0 is header)

            for col in range(n_index - 1, n_index):
                col_values = idx_df.iloc[:, col]
                last_val = None
                merge_start = start_row
                for row in range(n_rows):
                    val = col_values.iloc[row]
                    if val != last_val and row > 0:
                        if row + start_row - merge_start > 1:
                            worksheet.merge_range(
                                merge_start,
                                col,
                                row + start_row - 1,
                                col,
                                last_val,
                                wrap_format,
                            )
                        merge_start = row + start_row
                    last_val = val
                # Merge the last group
                if n_rows + start_row - merge_start > 1:
                    worksheet.merge_range(
                        merge_start,
                        col,
                        n_rows + start_row - 1,
                        col,
                        last_val,
                        wrap_format,
                    )

    return xlsxfile


def write_to_excel(df, filename="compliance-report.xlsx"):
    """
    Write a DataFrame to an Excel file with custom formatting using XlsxWriter.

    - Writes the DataFrame to the specified Excel file.
    - Applies custom header formatting.
    - Sets column widths and enables text wrapping for all columns.
    - Applies alternating row background color for readability.
    - Merges repeated MultiIndex cells for a cleaner look.

    Args:
        df (pd.DataFrame): The DataFrame to export.
        filename (str): The name of the Excel file to create.
    """
    # Create a Pandas Excel writer using XlsxWriter as the engine.

    with pd.ExcelWriter(filename, engine="xlsxwriter") as writer:
        df.to_excel(writer, index=True, sheet_name="catalog")
        workbook = writer.book
        worksheet = writer.sheets["catalog"]
        wrap_format = workbook.add_format(
            {"text_wrap": True, "align": "left", "valign": "top"}
        )
        grey_format = workbook.add_format(
            {"bg_color": "#F2F2F2", "text_wrap": True, "align": "left", "valign": "top"}
        )

        header_format = workbook.add_format(
            {
                "bold": True,
                "text_wrap": True,
                "valign": "top",
                "fg_color": "#D7E4BC",
                "border": 1,
            }
        )

        n_index = len(df.index.names)
        n_rows = len(df)
        n_cols = len(df.columns)

        # Write the column headers with the defined format.
        for col_num, value in enumerate(df.index.names):
            worksheet.write(0, col_num, value, header_format)

        # Write the data columns headers with the defined format.
        for col_num, value in enumerate(df.columns):
            worksheet.write(0, col_num + n_index, value, header_format)

        # Set wrap for all index columns
        for idx in range(n_index):
            worksheet.set_column(idx, idx, 30, wrap_format)

        # Set wrap for data columns
        for col_num in range(n_index, n_index + n_cols):
            worksheet.set_column(col_num, col_num, 50, wrap_format)

        # Apply alternating row color (starting after header row)
        for row in range(1, n_rows + 1):
            fmt = grey_format if row % 2 == 0 else wrap_format
            # worksheet.set_row(row, 60, fmt)
            # Write data columns
            for col in range(n_cols):
                value = df.iloc[row - 1, col]
                worksheet.write(row, col + n_index, value, fmt)

        # --- Merge repeated MultiIndex cells ---
        # Get the index values as a DataFrame
        idx_df = pd.DataFrame(df.index.tolist(), columns=df.index.names)
        start_row = 1  # Excel row index (0 is header)

        for col in range(n_index - 1, n_index):
            col_values = idx_df.iloc[:, col]
            last_val = None
            merge_start = start_row
            for row in range(n_rows):
                val = col_values.iloc[row]
                if val != last_val and row > 0:
                    if row + start_row - merge_start > 1:
                        worksheet.merge_range(
                            merge_start,
                            col,
                            row + start_row - 1,
                            col,
                            last_val,
                            wrap_format,
                        )
                    merge_start = row + start_row
                last_val = val
            # Merge the last group
            if n_rows + start_row - merge_start > 1:
                worksheet.merge_range(
                    merge_start, col, n_rows + start_row - 1, col, last_val, wrap_format
                )

In [10]:
write_to_excel(human_readable(df))

In [28]:
create_excel("report/compliance-report.csv")

CNRM-MF
GERICS
ICTP
KNMI


'report/compliance-report.xlsx'

In [18]:
df = pd.read_csv("report/compliance-report.csv")
human_readable(df).to_excel("report/compliance-report.xlsx", index=True)

In [ ]:
df